# F09: Cross-Domain Enterprise Dataset (Fabric Scenario)

## What You'll Learn

Enterprise data lakehouses don't serve a single department — they unify data
across retail operations, human resources, insurance, finance, and more. Building
and testing a lakehouse at this scale requires synthetic data that spans multiple
business domains with realistic volumes and relationships.

In this notebook you will:

1. **Understand** the enterprise data lakehouse scenario
2. **Compose** Retail, HR, and Insurance domains into a single dataset
3. **Generate** the composite at small scale
4. **Export** each domain as a star schema
5. **Write** partitioned Parquet files for lakehouse loading
6. **Review** the complete table inventory with row counts

## The Enterprise Lakehouse Pattern

```
Lakehouse/
  Tables/
    retail/
      dim_customer/
      dim_product/
      fact_order/
    hr/
      dim_employee/
      dim_department/
      fact_payroll/
    insurance/
      dim_policyholder/
      dim_policy/
      fact_claim/
```

## Prerequisites

- `sqllocks-spindle` installed
- Familiarity with composite domains (see T13)

## Time Estimate

**~15 minutes**

In [1]:
# Uncomment the line below if sqllocks-spindle is not yet installed.
# %pip install sqllocks-spindle

print("Installation cell ready. Uncomment the pip line above if needed.")

Installation cell ready. Uncomment the pip line above if needed.


## Step 1 — Compose Three Enterprise Domains

**What we're about to do:** Import the Retail, HR, and Insurance domains
and combine them into a `CompositeDomain`. This represents a mid-size
enterprise that sells products, employs people, and carries insurance.

**Why this matters:** Each domain brings its own schema — customers, orders,
and products from Retail; employees, departments, and payroll from HR;
policyholders, policies, and claims from Insurance. The composite ensures
all three are generated with coordinated seeds and prefixed table names.

**What to expect:** A composite domain listing all tables from all three
domains.

In [2]:
from sqllocks_spindle import Spindle
from sqllocks_spindle.domains.retail import RetailDomain
from sqllocks_spindle.domains.hr import HrDomain
from sqllocks_spindle.domains.insurance import InsuranceDomain
from sqllocks_spindle.domains.composite import CompositeDomain

retail = RetailDomain()
hr = HrDomain()
insurance = InsuranceDomain()

composite = CompositeDomain(domains=[retail, hr, insurance])

print("Enterprise Composite Domain")
print("=" * 50)

# Use child_domains property and get_schema() to list tables
total_tables = 0
for domain in composite.child_domains:
    schema = domain.get_schema()
    table_names = list(schema.tables.keys())
    total_tables += len(table_names)
    print(f"\n  {domain.name.upper()} ({len(table_names)} tables)")
    for t in table_names:
        print(f"    - {domain.name}_{t}")

print(f"\nTotal tables across all domains: {total_tables}")

Enterprise Composite Domain

  RETAIL (9 tables)
    - retail_customer
    - retail_address
    - retail_product_category
    - retail_product
    - retail_store
    - retail_promotion
    - retail_order
    - retail_order_line
    - retail_return

  HR (9 tables)
    - hr_department
    - hr_position
    - hr_employee
    - hr_compensation
    - hr_performance_review
    - hr_time_off_request
    - hr_training
    - hr_training_enrollment
    - hr_termination

  INSURANCE (9 tables)
    - insurance_agent
    - insurance_policyholder
    - insurance_policy_type
    - insurance_policy
    - insurance_coverage
    - insurance_claim
    - insurance_claim_payment
    - insurance_premium_payment
    - insurance_underwriting

Total tables across all domains: 27


## Step 2 — Generate at Small Scale

**What we're about to do:** Generate the full composite dataset at `small`
scale. This produces enough data for meaningful analytics while keeping
generation time fast.

**Why this matters:** Small scale is ideal for development and testing. For
load testing or performance benchmarking, you would bump this to `medium`
or `large` — the same code works at any scale.

**What to expect:** A dataset spanning all three domains with prefixed table
names and realistic row counts.

In [3]:
result = Spindle().generate(domain=composite, scale="small", seed=42)

print(f"Total tables: {len(result.tables)}")
total_rows = sum(len(df) for df in result.tables.values())
print(f"Total rows:   {total_rows:,}")

print(f"\n=== Table Inventory ===")
for name, df in result.tables.items():
    print(f"  {name:<35} {len(df):>8,} rows x {len(df.columns)} cols")

Total tables: 27
Total rows:   51,265

=== Table Inventory ===
  hr_position                               80 rows x 6 cols
  hr_training                              100 rows x 5 cols
  insurance_policy_type                     30 rows x 5 cols
  retail_customer                        1,000 rows x 8 cols
  retail_address                         1,500 rows x 10 cols
  retail_product_category                   50 rows x 4 cols
  retail_product                           500 rows x 6 cols
  retail_promotion                         300 rows x 6 cols
  retail_store                             150 rows x 5 cols
  hr_department                             30 rows x 5 cols
  hr_employee                              500 rows x 12 cols
  hr_compensation                        1,500 rows x 6 cols
  hr_performance_review                  1,250 rows x 7 cols
  hr_termination                            75 rows x 6 cols
  hr_time_off_request                    2,500 rows x 7 cols
  hr_training_enroll

## Step 3 — Domain-Level Summary

**What we're about to do:** Break down the dataset by domain to see how
data is distributed across the enterprise. We'll compute row counts,
table counts, and column counts per domain.

**Why this matters:** In a real lakehouse, each domain maps to a schema
or folder. Understanding the relative size of each domain helps with
capacity planning, partition strategies, and cost estimation.

**What to expect:** A summary table showing each domain's contribution
to the overall dataset.

In [4]:
# Group by domain prefix
domain_stats = {}
for table_name, df in result.tables.items():
    prefix = table_name.split("_")[0]
    if prefix not in domain_stats:
        domain_stats[prefix] = {"tables": 0, "rows": 0, "columns": 0}
    domain_stats[prefix]["tables"] += 1
    domain_stats[prefix]["rows"] += len(df)
    domain_stats[prefix]["columns"] += len(df.columns)

print("=== Enterprise Dataset by Domain ===")
print(f"{'Domain':<15} {'Tables':>8} {'Rows':>12} {'Columns':>10} {'% of Rows':>10}")
print("-" * 58)
for domain_name, stats in domain_stats.items():
    pct = stats['rows'] / total_rows * 100
    print(f"  {domain_name:<13} {stats['tables']:>8} {stats['rows']:>12,} {stats['columns']:>10} {pct:>9.1f}%")
print("-" * 58)
print(f"  {'TOTAL':<13} {len(result.tables):>8} {total_rows:>12,}")

=== Enterprise Dataset by Domain ===
Domain            Tables         Rows    Columns  % of Rows
----------------------------------------------------------
  hr                   9        8,035         61      15.7%
  insurance            9       21,380         65      41.7%
  retail               9       21,850         60      42.6%
----------------------------------------------------------
  TOTAL               27       51,265


## Step 4 — Export Star Schemas

**What we're about to do:** Transform each domain's 3NF tables into
star schemas — producing dimension and fact tables ready for Power BI
DirectLake or any analytical query engine.

**Why this matters:** A lakehouse gold layer is almost always modeled as
star schemas. By transforming each domain independently, we produce clean
dimension/fact models that map to Power BI datasets. Each domain gets its
own `dim_date` plus domain-specific dimensions and facts.

**What to expect:** Star schema summaries for each domain showing dimensions
and facts with row counts.

In [5]:
from sqllocks_spindle.transform import StarSchemaTransform

transform = StarSchemaTransform()

# Transform each domain's tables into a star schema
domain_instances = {"retail": retail, "hr": hr, "insurance": insurance}

stars = {}
for domain_name, domain_obj in domain_instances.items():
    # Extract this domain's tables (strip prefix for transform)
    domain_tables = {}
    prefix = f"{domain_name}_"
    for table_name, df in result.tables.items():
        if table_name.startswith(prefix):
            unprefixed = table_name[len(prefix):]
            domain_tables[unprefixed] = df

    star = transform.transform(domain_tables, domain_obj.star_schema_map())
    stars[domain_name] = star

    print(f"\n=== {domain_name.upper()} Star Schema ===")
    print(f"  Dimensions:")
    for name, df in star.dimensions.items():
        print(f"    {name:<25} {len(df):>8,} rows")
    print(f"  Facts:")
    for name, df in star.facts.items():
        print(f"    {name:<25} {len(df):>8,} rows")
    print(f"  dim_date: {len(star.date_dim):,} days")


=== RETAIL Star Schema ===
  Dimensions:
    dim_customer                 1,000 rows
    dim_product                    500 rows
    dim_store                      150 rows
    dim_promotion                  300 rows
  Facts:
    fact_sale                   12,500 rows
    fact_return                    850 rows
  dim_date: 2,054 days

=== HR Star Schema ===
  Dimensions:
    dim_employee                   500 rows
    dim_department                  30 rows
    dim_position                    80 rows
  Facts:
    fact_compensation            1,500 rows
    fact_performance             1,250 rows
    fact_time_off                2,500 rows
  dim_date: 1,460 days

=== INSURANCE Star Schema ===
  Dimensions:
    dim_policyholder             1,000 rows
    dim_agent                      100 rows
    dim_policy_type                 30 rows
    dim_policy                   1,800 rows
  Facts:
    fact_claim                     540 rows
    fact_claim_payment             810 rows
    fact_p

## Step 5 — Write Partitioned Parquet

**What we're about to do:** Export the star schemas as partitioned Parquet
files organized by domain — the exact format a Fabric Lakehouse expects.

**Why this matters:** Parquet is the native format for Fabric Lakehouses
and Delta tables. Writing domain-partitioned Parquet gives you files that
can be uploaded directly to a Lakehouse `Tables/` folder or registered as
Delta tables via Spark.

**What to expect:** Parquet files written to a domain-organized folder
structure with file sizes.

In [6]:
import os

output_base = "./f09_enterprise"

total_files = 0
total_size = 0

for domain_name, star in stars.items():
    domain_dir = os.path.join(output_base, domain_name)
    os.makedirs(domain_dir, exist_ok=True)

    # Write dimensions
    for name, df in star.dimensions.items():
        path = os.path.join(domain_dir, f"{name}.parquet")
        df.to_parquet(path, index=False)
        size = os.path.getsize(path)
        total_files += 1
        total_size += size

    # Write facts
    for name, df in star.facts.items():
        path = os.path.join(domain_dir, f"{name}.parquet")
        df.to_parquet(path, index=False)
        size = os.path.getsize(path)
        total_files += 1
        total_size += size

    # Write date dimension (shared but written per domain)
    path = os.path.join(domain_dir, "dim_date.parquet")
    star.date_dim.to_parquet(path, index=False)
    total_files += 1
    total_size += os.path.getsize(path)

print(f"=== Parquet Export Summary ===")
print(f"  Output directory: {output_base}")
print(f"  Total files:      {total_files}")
print(f"  Total size:       {total_size:,} bytes ({total_size/1024:.1f} KB)")

# Show directory structure
print(f"\n=== Directory Structure ===")
for root, dirs, files in os.walk(output_base):
    level = root.replace(output_base, "").count(os.sep)
    indent = "  " * (level + 1)
    folder = os.path.basename(root)
    print(f"{indent}{folder}/")
    sub_indent = "  " * (level + 2)
    for f in sorted(files):
        fpath = os.path.join(root, f)
        fsize = os.path.getsize(fpath)
        print(f"{sub_indent}{f:<35} {fsize:>8,} bytes")

=== Parquet Export Summary ===
  Output directory: ./f09_enterprise
  Total files:      22
  Total size:       1,591,095 bytes (1553.8 KB)

=== Directory Structure ===
  f09_enterprise/
    hr/
      dim_date.parquet                      25,678 bytes
      dim_department.parquet                 5,139 bytes
      dim_employee.parquet                  42,705 bytes
      dim_position.parquet                   7,625 bytes
      fact_compensation.parquet             68,542 bytes
      fact_performance.parquet              66,045 bytes
      fact_time_off.parquet                110,542 bytes
    insurance/
      dim_agent.parquet                     12,286 bytes
      dim_date.parquet                      25,678 bytes
      dim_policy.parquet                    95,768 bytes
      dim_policy_type.parquet                5,373 bytes
      dim_policyholder.parquet              96,105 bytes
      fact_claim.parquet                    36,235 bytes
      fact_claim_payment.parquet            56,650

      fact_return.parquet                   55,352 bytes
      fact_sale.parquet                    408,511 bytes


## Step 6 — Complete Table Inventory

**What we're about to do:** Print the full inventory of every table in the
enterprise dataset — both the source 3NF tables and the star schema outputs.
This is the complete catalog of what was generated.

**Why this matters:** When setting up a lakehouse, you need to know exactly
what tables exist, how many rows they contain, and how they relate. This
inventory is your starting point for creating a Fabric Lakehouse schema,
Power BI semantic model, or data catalog entries.

**What to expect:** A comprehensive table listing organized by domain and
layer (source vs. star schema).

In [7]:
print("=" * 70)
print("ENTERPRISE DATASET — COMPLETE TABLE INVENTORY")
print("=" * 70)

grand_total_rows = 0

for domain_name in ["retail", "hr", "insurance"]:
    print(f"\n{'─' * 70}")
    print(f"  {domain_name.upper()} DOMAIN")
    print(f"{'─' * 70}")

    # Source tables
    prefix = f"{domain_name}_"
    source_tables = {k: v for k, v in result.tables.items() if k.startswith(prefix)}
    source_rows = sum(len(df) for df in source_tables.values())

    print(f"\n  Source (3NF): {len(source_tables)} tables, {source_rows:,} rows")
    for name, df in source_tables.items():
        print(f"    {name:<35} {len(df):>8,} rows x {len(df.columns):>2} cols")

    # Star schema tables
    star = stars[domain_name]
    star_rows = sum(len(df) for df in star.dimensions.values()) + sum(len(df) for df in star.facts.values())

    print(f"\n  Star Schema: {len(star.dimensions) + len(star.facts)} tables, {star_rows:,} rows")
    for name, df in star.dimensions.items():
        print(f"    {name:<35} {len(df):>8,} rows  [dim]")
    for name, df in star.facts.items():
        print(f"    {name:<35} {len(df):>8,} rows  [fact]")

    grand_total_rows += source_rows

print(f"\n{'=' * 70}")
print(f"  GRAND TOTAL: {len(result.tables)} source tables, {grand_total_rows:,} rows")
print(f"{'=' * 70}")

ENTERPRISE DATASET — COMPLETE TABLE INVENTORY

──────────────────────────────────────────────────────────────────────
  RETAIL DOMAIN
──────────────────────────────────────────────────────────────────────

  Source (3NF): 9 tables, 21,850 rows
    retail_customer                        1,000 rows x  8 cols
    retail_address                         1,500 rows x 10 cols
    retail_product_category                   50 rows x  4 cols
    retail_product                           500 rows x  6 cols
    retail_promotion                         300 rows x  6 cols
    retail_store                             150 rows x  5 cols
    retail_order                           5,000 rows x  8 cols
    retail_order_line                     12,500 rows x  8 cols
    retail_return                            850 rows x  5 cols

  Star Schema: 6 tables, 15,300 rows
    dim_customer                           1,000 rows  [dim]
    dim_product                              500 rows  [dim]
    dim_store       

## What's Next?

You've built a complete enterprise dataset spanning three business domains,
transformed each into star schemas, and exported everything as partitioned
Parquet — ready for a Fabric Lakehouse.

| Notebook | What You'll Learn |
|----------|-------------------|
| **F10: Month-End Close** | Add time dimension with 12-month financial snapshots |
| **T13: Composite Domains** | Deep dive into composite domain mechanics and presets |
| **F01: Medallion Architecture** | Build bronze/silver/gold layers from this data |
| **F06: Semantic Model** | Create a Power BI semantic model from the star schemas |

**Key takeaways:**
- `CompositeDomain` with three domains produces a realistic enterprise dataset in one call
- Table prefixes (`retail_`, `hr_`, `insurance_`) prevent naming collisions
- `StarSchemaTransform` converts each domain's 3NF tables into dimension/fact models
- Parquet export with domain partitioning maps directly to Fabric Lakehouse structure
- Scale from `small` (dev/test) to `large` (load testing) with a single parameter change